# 07casestudy: the charts for the case study

PF9, step 8. Redraws the three charts in docs/case-study.html from the processed data, with the case study's font (Bricolage Grotesque). Run after 03moorings, 04hycommonthly and 06seaice.

- **cs_flowice.png**: yearly measured flow through the strait (UW moorings) above the sea ice in the map area (NSIDC), 1978 to 2030.
- **cs_heatice.png**: the key finding, one dot per year: heat carried north through the strait from June to November against the November sea ice.
- **cs_model.png**: the model (HYCOM) against the moorings for every month with measurements in the same month.

Needs fonttools and brotli for the font (`uv pip install fonttools brotli`).

## Section 0. Settings

Lines marked `# <-` can be changed.

In [ ]:
from pathlib import Path

PROJECT_DIR = Path.cwd()
DATA = PROJECT_DIR / "data"  # <- uw_annual.csv, uw_monthly.csv, nsidc_ice_monthly.csv, model_vs_moorings.csv
FONT_DIR = PROJECT_DIR / "docs" / "fonts"  # <- the case study's woff2 fonts
TTF_DIR = PROJECT_DIR / "data" / "fonts_ttf"  # <- converted fonts for matplotlib (made below)
OUT = PROJECT_DIR / "docs" / "images"  # <- where the charts are saved
DPI = 200  # <- resolution of the saved charts

HEAD = "#0f5f5c"    # <- headings: Bering teal
FLOW = "#0e6fa0"    # <- measured flow
MODEL = "#6b4fbf"   # <- model
ICE = "#5d7f99"     # <- sea ice
WARM = "#c2410c"    # <- too warm
MUTED = "#5b6474"

## Section 1. Libraries and the font

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm
from fontTools.ttLib import TTFont

OUT.mkdir(parents=True, exist_ok=True)
TTF_DIR.mkdir(parents=True, exist_ok=True)
for w in FONT_DIR.glob("bricolage*.woff2"):
    ttf = TTF_DIR / (w.stem + ".ttf")
    if not ttf.exists():
        f = TTFont(w)
        f.flavor = None
        f.save(ttf)
    fm.fontManager.addfont(str(ttf))
FAMILY = fm.FontProperties(fname=str(TTF_DIR / "bricolage-grotesque-latin-600-normal.ttf")).get_name()
plt.rcParams.update({"font.family": FAMILY, "font.size": 11, "axes.edgecolor": "#9aa3ad", "axes.labelcolor": "#1d2433",
                     "xtick.color": "#5b6474", "ytick.color": "#5b6474", "axes.spines.top": False, "axes.spines.right": False})
print("Font:", FAMILY)

## Section 2. Flow up, ice down

In [ ]:
ann = pd.read_csv(DATA / "uw_annual.csv")
ice = pd.read_csv(DATA / "nsidc_ice_monthly.csv")
a = ann.dropna(subset=["transport_sv"])
slope, icpt = np.polyfit(a["year"], a["transport_sv"], 1)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7.8, 3.7), sharex=True, gridspec_kw={"height_ratios": [1, 1], "hspace": 0.18})
ax1.axvspan(1992, 1997.5, color="#eef1f4", lw=0)
ax1.text(1994.7, 1.37, "no data", ha="center", fontsize=8.5, color=MUTED)
ax1.plot(ann["year"], ann["transport_sv"], "-", color=FLOW, lw=1.3)  # NaN rows break the line at the data gap
ax1.errorbar(a["year"], a["transport_sv"], yerr=a["transport_err_sv"], fmt="o", color=FLOW, ms=3.5, elinewidth=0.8, capsize=0)
xs = np.array([1991, 2024])
ax1.plot(xs, icpt + slope * xs, "--", color=HEAD, lw=1.2)
ax1.text(2024.6, icpt + slope * 2024, f"+{slope * 10:.3f} Sv\nper decade", fontsize=8.5, color=HEAD, va="center")
pk = a.loc[a["transport_sv"].idxmax()]
ax1.annotate(f"{int(pk.year)}: {pk.transport_sv:.2f}", (pk.year, pk.transport_sv), xytext=(-46, 6), textcoords="offset points", fontsize=8.5, color=FLOW)
ax1.set_ylim(0.5, 1.45)
ax1.set_ylabel("Sv", fontsize=9.5)
ax1.set_title("Yearly mean flow through the Bering Strait (northward)", loc="left", fontsize=10.5, color="#1d2433", pad=4)

months = {11: ("November", ICE, "-"), 5: ("May", "#93aec2", "-"), 8: ("August", "#b9cad6", "-")}
for m, (name, col, ls) in months.items():
    s = ice[ice["month"] == m].sort_values("year")
    s = s[s["year"] <= 2025]
    ax2.plot(s["year"], s["ice_area_km2"] / 1000, ls, color=col, lw=1.4 if m == 11 else 1.1)
    ax2.text(s["year"].iloc[-1] + 0.6, s["ice_area_km2"].iloc[-1] / 1000, name, fontsize=8.5, color=col, va="center")
ax2.set_ylim(0, 700)
ax2.set_ylabel("1000 km²", fontsize=9.5)
ax2.set_title("Sea ice in the map area, monthly mean (satellite)", loc="left", fontsize=10.5, color="#1d2433", pad=4)
ax2.set_xlim(1978, 2031)
for ax in (ax1, ax2):
    ax.tick_params(labelsize=8.5)
    ax.grid(axis="y", color="#e6e8ec", lw=0.7)
fig.savefig(OUT / "cs_flowice.png", dpi=DPI, bbox_inches="tight", facecolor="white")
plt.show()

## Section 3. Model against measured

In [ ]:
mv = pd.read_csv(DATA / "model_vs_moorings.csv")
same = mv[mv["measured_kind"] != "clim"]
fig, (b1, b2) = plt.subplots(1, 2, figsize=(7.8, 3.3), gridspec_kw={"wspace": 0.32})
mk = {2: "o", 5: "s", 8: "^", 11: "D"}
mname = {2: "Feb", 5: "May", 8: "Aug", 11: "Nov"}
for m in [2, 5, 8, 11]:
    d = same[same["month"] == m]
    for era, filled in [(d["year"] <= 2015, True), (d["year"] >= 2016, False)]:
        e = d[era]
        kw = dict(marker=mk[m], s=26, linewidths=1.1, edgecolors=MODEL)
        b1.scatter(e["uw_transport_sv"], e["transport_sv"], facecolors=MODEL if filled else "white", **kw)
        b2.scatter(e["uw_temp_c"], e["a3_bottom_temp_c"], facecolors=MODEL if filled else "white", **kw)
lim = (-0.5, 2.3)
b1.plot(lim, lim, color="#9aa3ad", lw=0.9)
b1.set_xlim(lim); b1.set_ylim(lim)
b1.set_xlabel("measured (Sv)", fontsize=9.5); b1.set_ylabel("model (Sv)", fontsize=9.5)
b1.set_title("Flow through the strait, same month", loc="left", fontsize=10.5, pad=4)
south = same[same["transport_sv"] < 0]
for _, r in south.iterrows():
    b1.annotate(f"{mname[int(r.month)]} {int(r.year)}: both south", (r.uw_transport_sv, r.transport_sv), xytext=(8, -4), textcoords="offset points", fontsize=8, color=MUTED)
tl = (-2.5, 7.0)
b2.plot(tl, tl, color="#9aa3ad", lw=0.9)
b2.fill_between(tl, [t + 1 for t in tl], [7.0, 7.0], color=WARM, alpha=0.06, lw=0)
b2.text(-2.2, 5.0, "model more than\n1 °C too warm", fontsize=8, color=WARM)
b2.set_xlim(tl); b2.set_ylim(tl)
b2.set_xlabel("measured (°C)", fontsize=9.5); b2.set_ylabel("model (°C)", fontsize=9.5)
b2.set_title("Water near the seabed at A3", loc="left", fontsize=10.5, pad=4)
for ax in (b1, b2):
    ax.tick_params(labelsize=8.5)
    ax.set_aspect("equal")
    ax.grid(color="#eef0f3", lw=0.6)
# legend: markers for months, filled/hollow for era
from matplotlib.lines import Line2D
h = [Line2D([], [], marker=mk[m], ls="", mfc=MODEL, mec=MODEL, ms=5, label=mname[m]) for m in [2, 5, 8, 11]]
h += [Line2D([], [], marker="o", ls="", mfc=MODEL, mec=MODEL, ms=5, label="reanalysis (2000 to 2015)"),
      Line2D([], [], marker="o", ls="", mfc="white", mec=MODEL, ms=5, label="analysis (2019 on)")]
fig.legend(handles=h, loc="lower center", ncol=6, fontsize=8.3, frameon=False, bbox_to_anchor=(0.5, -0.13), handletextpad=0.2, columnspacing=1.0)
fig.savefig(OUT / "cs_model.png", dpi=DPI, bbox_inches="tight", facecolor="white")
plt.show()

n = len(same)
err = (same["transport_sv"] - same["uw_transport_sv"])
print("same-month rows", n, "| years", sorted(same["year"].unique()))
print("flow: mean model - measured %.2f Sv, mean abs %.2f Sv" % (err.mean(), err.abs().mean()))
for era, sel in [("reanalysis", same["year"] <= 2015), ("analysis", same["year"] >= 2016)]:
    e = err[sel]
    print(f"  {era}: n={len(e)}, mean {e.mean():.2f}, mean abs {e.abs().mean():.2f}, within 0.2: {(e.abs() <= 0.2).sum()}")
print("bottom temp Aug model - measured:", (same[same.month == 8]["a3_bottom_temp_c"] - same[same.month == 8]["uw_temp_c"]).round(1).tolist())
print("trend", round(slope * 10, 3), "Sv per decade; line", round(icpt + slope * 1991, 2), "to", round(icpt + slope * 2024, 2))

## Section 4. Heat in, ice out (the key finding)

In [ ]:
mon = pd.read_csv(DATA / "uw_monthly.csv")
js = mon[mon["month"].between(6, 11)].groupby("year").agg(heat=("heat", "mean"), n=("heat", "count"))
js = js[js["n"] >= 5]
nov = ice[ice["month"] == 11].set_index("year")["ice_area_km2"] / 1000
d = js.join(nov.rename("nov"), how="inner").dropna()
r_raw = np.corrcoef(d["heat"], d["nov"])[0, 1]
yrs = d.index.values.astype(float)
res = lambda v: v - np.polyval(np.polyfit(yrs, v, 1), yrs)
r_det = np.corrcoef(res(d["heat"].values), res(d["nov"].values))[0, 1]
fig, ax = plt.subplots(figsize=(7.8, 3.9))
cmap = matplotlib.colors.LinearSegmentedColormap.from_list("t", ["#bfe0e3", HEAD])
sc = ax.scatter(d["heat"], d["nov"], c=d.index, cmap=cmap, s=46, edgecolors=HEAD, linewidths=0.6, zorder=3)
p = np.polyfit(d["heat"], d["nov"], 1)
xx = np.linspace(d["heat"].min() - 0.5, d["heat"].max() + 0.5, 10)
ax.plot(xx, np.polyval(p, xx), "--", color="#9aa3ad", lw=1.0, zorder=2)
label_off = {1991: (-30, -10), 2001: (6, 2), 2012: (6, 2), 1999: (6, 2), 2007: (-12, 8), 2017: (-14, -13), 2018: (6, 3), 2019: (6, 2), 2020: (6, -3), 2023: (6, 3), 2024: (6, 3)}
for y, row in d.iterrows():
    if y in label_off:
        ax.annotate(str(y), (row.heat, row.nov), xytext=label_off[y], textcoords="offset points", fontsize=8, color=MUTED)
ax.set_xlabel("Heat carried north through the strait, June to November mean (TW)", fontsize=9.5)
ax.set_ylabel("November sea ice in the map area\n(1000 km²)", fontsize=9.5)
ax.set_title("Warmer inflow, later freeze-up: one dot per year", loc="left", fontsize=10.5, pad=4)
ax.text(0.99, 0.95, f"r = {r_raw:.2f} ({len(d)} years)\nr = {r_det:.2f} with both trends removed", transform=ax.transAxes, ha="right", va="top", fontsize=8.5, color=MUTED)
cb = fig.colorbar(sc, ax=ax, pad=0.015, fraction=0.03)
cb.ax.tick_params(labelsize=8)
cb.outline.set_visible(False)
ax.set_ylim(0, 620)
ax.tick_params(labelsize=8.5)
ax.grid(color="#eef0f3", lw=0.6)
fig.savefig(OUT / "cs_heatice.png", dpi=DPI, bbox_inches="tight", facecolor="white")
plt.show()
print(f"heat-ice: {len(d)} years {int(d.index.min())}-{int(d.index.max())}, r {r_raw:.2f}, detrended {r_det:.2f}, r2 {r_raw**2:.2f}/{r_det**2:.2f}")
print("top heat years:", d.sort_values("heat", ascending=False).head(5).round(1).to_dict("index"))

___
**END**